Titanic: Machine Learning from Disaster

In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


WITH JAX 

In [17]:
# ============================================
# TITANIC LOGISTIC REGRESSION - FULL JAX
# ============================================

# -----------------------------
# IMPORTS
# -----------------------------
import pandas as pd
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap
import time

# -----------------------------
# MEMBER 1: DATA LOADING & PREPROCESSING
# -----------------------------
path = "/kaggle/input/competitions/titanic/"
train = pd.read_csv(path + "train.csv")

# Fill missing values
train["Age"] = train["Age"].fillna(train["Age"].median())
train["Fare"] = train["Fare"].fillna(train["Fare"].median())
train["Embarked"] = train["Embarked"].fillna(train["Embarked"].mode()[0])

# Drop irrelevant columns
train = train.drop(["PassengerId","Name","Ticket","Cabin"], axis=1)

# New features
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
train["IsAlone"] = (train["FamilySize"] == 1).astype(int)

# One-hot encode categorical features
train = pd.get_dummies(train, columns=["Sex","Embarked"], drop_first=True)

# Define feature list
features = ["Pclass","Age","SibSp","Parch","Fare",
            "Sex_male","Embarked_Q","Embarked_S",
            "FamilySize","IsAlone"]

X = train[features].values.astype(jnp.float32)
y = train["Survived"].values.reshape(-1,1).astype(jnp.float32)

# Normalize features
X_mean = X.mean(axis=0)
X_std = X.std(axis=0) + 1e-8
X = (X - X_mean)/X_std
X = jnp.array(X)
y = jnp.array(y)

print("Data preprocessing done. X shape:", X.shape, "y shape:", y.shape)

# -----------------------------
# MEMBER 2: MODEL DEFINITION
# -----------------------------
def sigmoid(z):
    return 1/(1+jnp.exp(-z))

# Single prediction
def predict_single(w, x):
    return sigmoid(jnp.dot(x, w))

# Batch prediction with vmap
predict_batch = vmap(predict_single, in_axes=(None,0))

# Cross-entropy loss
def loss_fn(w, X, y):
    preds = predict_batch(w, X)
    return -jnp.mean(y*jnp.log(preds+1e-8) + (1-y)*jnp.log(1-preds+1e-8))

# -----------------------------
# MEMBER 3: TRAINING
# -----------------------------
grad_fn = grad(loss_fn)

@jit
def update(w, X, y, lr):
    return w - lr * grad_fn(w,X,y)

@jit
def train_loop(w, X, y, lr, steps):
    def body(i,w):
        return update(w,X,y,lr)
    return jax.lax.fori_loop(0, steps, body, w)

# Initialize weights
w = jnp.zeros((X.shape[1],1))

# Train
start = time.time()
w = train_loop(w, X, y, lr=0.05, steps=5000)
end = time.time()
print("Training done in", end-start,"seconds")

# Training accuracy
train_preds = predict_batch(w, X)
train_preds_binary = (train_preds > 0.5).astype(int)
train_acc = jnp.mean((train_preds_binary == y).astype(jnp.float32))
print("Training Accuracy:", train_acc)

# -----------------------------
# MEMBER 4: TEST DATA PREPROCESSING & PREDICTION
# -----------------------------
test = pd.read_csv(path + "test.csv")
test_ids = test["PassengerId"]

# Fill missing numerical values
test["Age"] = test["Age"].fillna(train["Age"].median())
test["Fare"] = test["Fare"].fillna(train["Fare"].median())

# Drop unnecessary columns
for col in ["PassengerId","Name","Ticket","Cabin"]:
    if col in test.columns:
        test = test.drop(col, axis=1)

# New features
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

# One-hot encode categorical columns
categorical_cols = []
for col in ["Sex","Embarked"]:
    if col in test.columns:
        categorical_cols.append(col)
if len(categorical_cols)>0:
    test = pd.get_dummies(test, columns=categorical_cols, drop_first=True)

# Align test columns with training features
for col in features:
    if col not in test.columns:
        test[col] = 0
test = test[features]

# Convert to float32 & normalize
X_test = test.values.astype(jnp.float32)
X_test = (X_test - X_mean)/X_std
X_test = jnp.array(X_test)

# Predict
test_preds = predict_batch(w, X_test)
test_preds_binary = (test_preds > 0.5).astype(int)

# Submission CSV
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": test_preds_binary.flatten()
})
submission.to_csv("submission.csv", index=False)
print("Submission created ")
submission.head()

# -----------------------------
# REFLECTION ANSWERS
# -----------------------------
print("\nReflection:")
print(" vmap is faster than loops because it vectorizes computations across all samples using XLA.")
print(" Combining jit + grad improves training speed by compiling the gradient updates into optimized machine code.")

Data preprocessing done. X shape: (891, 10) y shape: (891, 1)
Training done in 0.1574113368988037 seconds
Training Accuracy: 0.7923682
Submission created 

Reflection:
 vmap is faster than loops because it vectorizes computations across all samples using XLA.
 Combining jit + grad improves training speed by compiling the gradient updates into optimized machine code.


In [18]:
# ============================================
# TITANIC LOGISTIC REGRESSION - FINAL (NO ERRORS)
# ============================================

# -----------------------------
# IMPORTS
# -----------------------------
import pandas as pd
import jax
import jax.numpy as jnp
from jax import grad, jit, lax
import time

# -----------------------------
# LOAD DATA
# -----------------------------
path = "/kaggle/input/competitions/titanic/"
train = pd.read_csv(path + "train.csv")

# -----------------------------
# PREPROCESSING (TRAIN)
# -----------------------------

# Fill missing values
train["Age"] = train["Age"].fillna(train["Age"].median())
train["Fare"] = train["Fare"].fillna(train["Fare"].median())

#  SAVE EMBARKED MODE (VERY IMPORTANT FIX)
if "Embarked" in train.columns:
    embarked_mode = train["Embarked"].mode()[0]
    train["Embarked"] = train["Embarked"].fillna(embarked_mode)
else:
    embarked_mode = "S"

# Feature engineering
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
train["IsAlone"] = (train["FamilySize"] == 1).astype(int)

# Title extraction (FIXED regex)
train["Title"] = train["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
train["Title"] = train["Title"].replace(
    ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'],
    'Rare'
)
train["Title"] = train["Title"].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})

# Drop unused columns
train = train.drop(["PassengerId","Name","Ticket","Cabin"], axis=1)

# One-hot encoding (SAFE)
cols = []
for col in ["Sex", "Embarked", "Title"]:
    if col in train.columns:
        cols.append(col)

train = pd.get_dummies(train, columns=cols, drop_first=True)

# Features
features = train.drop("Survived", axis=1).columns

# Convert to arrays
X = train[features].values.astype(jnp.float32)
y = train["Survived"].values.reshape(-1,1).astype(jnp.float32)

# Normalize
X_mean = X.mean(axis=0)
X_std = X.std(axis=0) + 1e-8
X = (X - X_mean)/X_std

X = jnp.array(X)
y = jnp.array(y)

print("Train preprocessing done:", X.shape)

# -----------------------------
# MODEL
# -----------------------------
def sigmoid(z):
    return 1 / (1 + jnp.exp(-z))

def predict(w, X):
    return sigmoid(jnp.dot(X, w))

def loss_fn(w, X, y, lam=0.01):
    preds = predict(w, X)
    loss = -jnp.mean(y*jnp.log(preds+1e-8) + (1-y)*jnp.log(1-preds+1e-8))
    reg = lam * jnp.sum(w**2)
    return loss + reg

grad_fn = grad(loss_fn)

@jit
def update(w, X, y, lr):
    return w - lr * grad_fn(w, X, y)

@jit
def train_model(w, X, y, lr, steps):
    def body(i, w):
        return update(w, X, y, lr)
    return lax.fori_loop(0, steps, body, w)

# Initialize weights
w = jnp.zeros((X.shape[1], 1))

# -----------------------------
# TRAIN
# -----------------------------
start = time.time()
w = train_model(w, X, y, lr=0.1, steps=1200)
end = time.time()

print("Training time:", end - start)

# Accuracy
preds = predict(w, X)
preds_binary = (preds > 0.5).astype(int)
acc = jnp.mean((preds_binary == y).astype(jnp.float32))

print("Training Accuracy:", acc)

# -----------------------------
# TEST DATA
# -----------------------------
test = pd.read_csv(path + "test.csv")
test_ids = test["PassengerId"]

# Fill missing safely
test["Age"] = test["Age"].fillna(train["Age"].median())
test["Fare"] = test["Fare"].fillna(train["Fare"].median())

#  USE SAVED MODE (FIXED)
if "Embarked" in test.columns:
    test["Embarked"] = test["Embarked"].fillna(embarked_mode)

# Feature engineering
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

# Title extraction
test["Title"] = test["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
test["Title"] = test["Title"].replace(
    ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'],
    'Rare'
)
test["Title"] = test["Title"].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})

# Drop unused columns safely
drop_cols = ["PassengerId","Name","Ticket","Cabin"]
for col in drop_cols:
    if col in test.columns:
        test = test.drop(col, axis=1)

# One-hot encoding (SAFE)
cols = []
for col in ["Sex", "Embarked", "Title"]:
    if col in test.columns:
        cols.append(col)

test = pd.get_dummies(test, columns=cols, drop_first=True)

# Align columns
for col in features:
    if col not in test.columns:
        test[col] = 0

test = test[features]

# Normalize
X_test = test.values.astype(jnp.float32)
X_test = (X_test - X_mean)/X_std
X_test = jnp.array(X_test)

# -----------------------------
# PREDICT
# -----------------------------
test_preds = predict(w, X_test)
test_preds_binary = (test_preds > 0.5).astype(int)

# Submission
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": test_preds_binary.flatten()
})

submission.to_csv("submission.csv", index=False)

print("Submission created ")
print(submission.head())

Train preprocessing done: (891, 14)
Training time: 0.17103791236877441
Training Accuracy: 0.81257015
Submission created 
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1


In [19]:
# ============================================
# TITANIC LOGISTIC REGRESSION - FAST & EFFICIENT
# ============================================

# -----------------------------
# IMPORTS
# -----------------------------
import pandas as pd
import jax
import jax.numpy as jnp
from jax import grad, jit
import time

# -----------------------------
# LOAD DATA
# -----------------------------
path = "/kaggle/input/competitions/titanic/"
train = pd.read_csv(path + "train.csv")
test = pd.read_csv(path + "test.csv")
test_ids = test["PassengerId"]

# -----------------------------
# PREPROCESSING FUNCTION
# -----------------------------
def preprocess(df, train_stats=None):
    # Fill missing
    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())
    
    # Embarked
    embarked_mode = df["Embarked"].mode()[0] if "Embarked" in df.columns else "S"
    if "Embarked" in df.columns:
        df["Embarked"] = df["Embarked"].fillna(embarked_mode)
    
    # Feature engineering
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    
    # Title extraction
    df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
    df["Title"] = df["Title"].replace(
        ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare'
    )
    df["Title"] = df["Title"].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})
    
    # Drop unused
    drop_cols = ["PassengerId","Name","Ticket","Cabin"]
    df = df.drop([col for col in drop_cols if col in df.columns], axis=1)
    
    # One-hot encoding
    for col in ["Sex","Embarked","Title"]:
        if col in df.columns:
            df = pd.get_dummies(df, columns=[col], drop_first=True)
    
    # Align columns with train
    if train_stats is not None:
        for col in train_stats["features"]:
            if col not in df.columns:
                df[col] = 0
        df = df[train_stats["features"]]
    
    # Normalize numeric features only
    numeric_cols = ["Pclass","Age","SibSp","Parch","Fare","FamilySize","IsAlone"]
    X = df.values.astype(jnp.float32)
    if train_stats is None:
        mean = X.mean(axis=0)
        std = X.std(axis=0)+1e-8
    else:
        mean = train_stats["mean"]
        std = train_stats["std"]
    numeric_idx = [df.columns.get_loc(c) for c in numeric_cols]
    X[:, numeric_idx] = (X[:, numeric_idx] - mean[numeric_idx]) / std[numeric_idx]
    
    return X, {"features": df.columns, "mean": mean, "std": std}

# -----------------------------
# PREPROCESS TRAIN
# -----------------------------
X, stats = preprocess(train)
y = train["Survived"].values.reshape(-1,1).astype(jnp.float32)
y = jnp.array(y)

print("Train preprocessing done:", X.shape)

# -----------------------------
# MODEL
# -----------------------------
def sigmoid(z):
    return 1 / (1 + jnp.exp(-z))

def predict(w, X):
    return sigmoid(X @ w)

def loss_fn(w, X, y, lam=0.01):
    p = predict(w, X)
    return -jnp.mean(y*jnp.log(p+1e-8) + (1-y)*jnp.log(1-p+1e-8)) + lam*jnp.sum(w**2)

grad_fn = jit(grad(loss_fn))

@jit
def update(w, X, y, lr):
    return w - lr * grad_fn(w, X, y)

# Initialize weights zeros
w = jnp.zeros((X.shape[1],1))

# -----------------------------
# TRAIN
# -----------------------------
lr = 0.1
steps = 400  # fast & efficient

start = time.time()
for _ in range(steps):
    w = update(w, X, y, lr)
end = time.time()
print("Training time:", end - start)

# Accuracy
preds = (predict(w, X) > 0.5).astype(jnp.int32)
acc = jnp.mean((preds == y).astype(jnp.float32))
print("Training Accuracy:", acc)

# -----------------------------
# PREPROCESS TEST
# -----------------------------
X_test, _ = preprocess(test, stats)
X_test = jnp.array(X_test)

# -----------------------------
# PREDICT & SUBMISSION
# -----------------------------
test_preds = predict(w, X_test)
test_preds_binary = (test_preds > 0.5).astype(int)

submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": test_preds_binary.flatten()
})
submission.to_csv("submission.csv", index=False)
print("Submission created ")
print(submission.head())

Train preprocessing done: (891, 15)
Training time: 0.14943170547485352
Training Accuracy: 0.97418636
Submission created 
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         0


WITH OUT JAX